<!-- nav -->
[← 2 · Answers you can trust](02_Answers_You_Can_Trust.ipynb) · [Home](../../README.md) · [5 · Deciding what you can now →](05_Deciding_What_You_Can_Now.ipynb)

# 3 · Constraints over collections

**The problem.** The latest reading isn't the whole story. A logger records the temperature every few minutes, and a
shipment that spent an hour at 12 °C may be back at 5 °C by the time it's checked. The real constraints are about *all*
the readings: every one in range, or at most two brief excursions, never above 12 °C. And about the loggers a shipment
carries: each one's battery must last the trip.

This case study writes constraints over lists and over an object's related objects, with **quantifiers**.

In [1]:
from mbse.Expressions import Evaluators, Expressions as E
from mbse.Expressions.Framework import Terms
from toolkit import cold_chain, show

store = cold_chain()
Expression, Term = store.Expressions.Framework.Expression, store.Expressions.Framework.Term
with store.transaction():
    steady = store.Shipment(lambda s: s.id("S1").readings([4.0, 5.5, 6.1, 5.0]))
    blip = store.Shipment(lambda s: s.id("S2").readings([4.0, 8.6, 9.1, 5.0]))
    cooked = store.Shipment(lambda s: s.id("S3").readings([4.0, 12.5, 14.0, 13.2]))
    silent = store.Shipment(lambda s: s.id("S4"))  # no readings uploaded
shipments = [steady, blip, cooked, silent]
shipment, c = E.variable("shipment"), E.variable("c")


def table(constraint):
    return {s.id.get(): Evaluators.OfAny(constraint, {"shipment": s}) for s in shipments}

## Step 1: a list is a collection

`readings` is a list property (a container, in mbse-schemas' terms). Reading it gives a **collection**, which a handful
of operations take: `count`, `item` (by position, from 0), `in`, `sum`, `min`, `max` and `unique`.

These quick questions aren't constraints to keep, so each is written as a term alone, `Term(writer)`: a value with no
`Expression` object around it, which evaluates the same way.

In [2]:
with store.transaction():
    questions = [Term(writer) for writer in [shipment.readings, shipment.readings.count(), shipment.readings.item(0),
                                             shipment.readings.max(), E.literal(9.1).in_(shipment.readings),
                                             shipment.readings.unique()]]
for expression in questions:
    print(f"{show(expression):40}", Evaluators.OfAny(expression, {"shipment": blip}))

get(shipment, 'readings')                [4.0, 8.6, 9.1, 5.0]
count(get(shipment, 'readings'))         4
item(get(shipment, 'readings'), 0)       4.0
max(get(shipment, 'readings'))           9.1
in(9.1, get(shipment, 'readings'))       True
unique(get(shipment, 'readings'))        True


Collections treat unknowns as everything else does. There's no item at position 9, so it's unknown,
not an error; the silent shipment has no readings at all, so everything about them is unknown:

In [3]:
with store.transaction():
    questions = [Term(writer) for writer in [shipment.readings.item(9), shipment.readings.count(), shipment.readings.max()]]
for expression in questions:
    print(f"{show(expression):40}", Evaluators.OfAny(expression, {"shipment": blip}), Evaluators.OfAny(expression, {"shipment": silent}))

item(get(shipment, 'readings'), 9)       None None
count(get(shipment, 'readings'))         4 None
max(get(shipment, 'readings'))           9.1 None


## Step 2: quantifiers

A **quantifier** binds a name to each item in turn, here `c`, and tests a body. `all` holds when the body holds for
every item, `any` when it holds for one, and `count` counts the items it holds for; writers spell them `.all`, `.any`
and `.count_where`:

In [4]:
with store.transaction():
    every_in_range = Expression(lambda e: e.root(shipment.readings.all("c", E.literal(2.0).le(c).and_(c.le(8.0)))))
print(show(every_in_range))
table(every_in_range)

all(get(shipment, 'readings'), c => and(le(2.0, c), le(c, 8.0)))


{'S1': True, 'S2': False, 'S3': False, 'S4': None}

The steady shipment passes and the others fail; the silent one is unknown. The quantifier is a term of its own
kind, which binds `c` the way a let does, so `c` means nothing outside the body.

The real constraint allows brief excursions: at most two readings above 8 °C, and none above 12 °C. That's a count and a
maximum:

In [5]:
excursions = shipment.readings.count_where("c", c.gt(8.0))  # a writer, not yet written
with store.transaction():
    allowed = Expression(lambda e: e.root(excursions.le(2).and_(shipment.readings.max().le(12.0))))
print(show(allowed))
table(allowed)

and(le(count(get(shipment, 'readings'), c => gt(c, 8.0)), 2), le(max(get(shipment, 'readings')), 12.0))


{'S1': True, 'S2': True, 'S3': False, 'S4': None}

The blip is allowed now; the cooked shipment still isn't.

## Step 3: empty collections

An empty collection is data too, and quantifiers answer the way logic does: every item of nothing satisfies anything
(`all` is true), none satisfies something (`any` is false), and the count is 0. What has no answer is the greatest of
nothing, so `max` is unknown, and a constraint that relies on it says so:

In [6]:
with store.transaction():
    empty = store.Shipment(lambda s: s.id("S5").readings([]))
    questions = {"every_in_range": every_in_range, "any above 8": Term(shipment.readings.any("c", c.gt(8.0))),
                 "excursions": Term(excursions), "max": Term(shipment.readings.max()), "allowed": allowed}
for name, constraint in questions.items():
    print(f"{name:16}", Evaluators.OfAny(constraint, {"shipment": empty}))

every_in_range   True
any above 8      False
excursions       0
max              None
allowed          None


`allowed` for an empty list is unknown, not true: `excursions` is 0, which is fine, but the maximum is unknown. A
shipment that logged nothing hasn't shown it stayed cold, and the constraint doesn't pretend otherwise. If an empty log
*should* pass, say so explicitly, with `count`:

In [7]:
with store.transaction():  # a writer builds on a writer or a term value, such as an expression's root
    lenient = Expression(lambda e: e.root(shipment.readings.count().eq(0).or_(Terms.term(allowed))))
print(Evaluators.OfAny(lenient, {"shipment": empty}), Evaluators.OfAny(lenient, {"shipment": cooked}))

True False


## Step 4: an object's related objects

A shipment carries loggers, through the relation `Carries`, whose entries record the slot each logger rides in: a
shipment's adjacency `loggers`. `entries(object, name)` gives an object's entries in an adjacency as a collection of
**records**, which `get` reads like objects: the linked logger, and the entry's own properties.

In [8]:
with store.transaction():
    l17 = store.Logger(lambda l: l.serial("L-17").battery(80))
    l22 = store.Logger(lambda l: l.serial("L-22").battery(15))
    trip = store.Shipment(lambda s: s.id("S6").readings([5.0])
                          .loggers([lambda e: e.slot(1).logger(l17), lambda e: e.slot(2).logger(l22)]))
    questions = [Term(writer) for writer in [shipment.entries("loggers").count(), shipment.entries("loggers").item(1).get("slot"),
                                             shipment.entries("loggers").item(1).get("logger").get("serial")]]
e = E.variable("e")
for expression in questions:
    print(f"{show(expression):70}", Evaluators.OfAny(expression, {"shipment": trip}))

count(entries(shipment, 'loggers'))                                    2
get(item(entries(shipment, 'loggers'), 1), 'slot')                     2
get(get(item(entries(shipment, 'loggers'), 1), 'logger'), 'serial')    L-22


`.get("logger")` is the writer of `get(e, 'logger')`; `e.logger` would be the same. So "every logger's battery
lasts the trip" is a quantifier over the entries:

In [9]:
with store.transaction():
    charged = Expression(lambda x: x.root(shipment.entries("loggers").all("e", e.logger.battery.ge(20))))
    flat = Term(shipment.entries("loggers").count_where("e", e.logger.battery.lt(20)))
print(show(charged))
print(Evaluators.OfAny(charged, {"shipment": trip}))
Evaluators.OfAny(flat, {"shipment": trip})

all(entries(shipment, 'loggers'), e => ge(get(get(e, 'logger'), 'battery'), 20))
False


1

## Why it works this way

- **Quantifiers instead of loops.** A loop can do anything, including never finish. A quantifier visits each item of
  data once, so a constraint with quantifiers is still bounded, total and pure, and still a constraint a solver or a
  hardware checker could take.
- **Collections come from data.** There are no list literals: a collection is a list property, or an object's entries,
  or a variable the caller binds. That keeps constraints about the data, not little programs of their own.
- **The empty case is honest.** `all` of nothing is true because nothing violates it; the maximum of nothing doesn't
  exist. Constraints that need an answer for the empty case say what it is.

<!-- nav -->
[← 2 · Answers you can trust](02_Answers_You_Can_Trust.ipynb) · [Home](../../README.md) · [5 · Deciding what you can now →](05_Deciding_What_You_Can_Now.ipynb)